# FIT3143 Applied #2, Task 1: Image rotation with CUDA

**Team:** Erwyna Soo Wen Xin (36555789, esoo0013@student.monash.edu) and
Taabish Farooq Bhat (35473932, ttaa0006@student.monash.edu)

This notebook compiles and runs `rotate_cuda.cu` on a Colab GPU, sweeps
block sizes and image sizes, and draws the graphs used in our slides.

**Before running:** Runtime > Change runtime type > **T4 GPU**, then
Runtime > Run all. The whole notebook takes a few minutes.

The Week 9 class notebook used Numba (`@cuda.jit`). Here the main program
is CUDA C++ compiled with `nvcc`, so the `<<<grid, block>>>` launch syntax
from the spec appears literally in the code. A short Numba version is at
the end to tie back to Week 9.

| Variant | Feature shown |
|---|---|
| v0 | CPU serial baseline (nearest, bilinear), also the reference answer |
| v1 | naive 1D grid, nearest neighbour |
| v2 | 2D grid of 2D blocks (`blockIdx`, `threadIdx`, `blockDim`) |
| v3 | bilinear interpolation (quality vs cost) |
| v4 | pageable host memory, compared with pinned in v2/v3 |
| v5 | CUDA streams: (a) one image in row chunks, (b) a batch of images |
| v6 | texture object with hardware bilinear filtering |

Every stage (H2D, kernel, D2H, total) is timed separately with
`cudaEvent`s, and every GPU output is compared with the CPU output.

## Results from our Colab run (Tesla T4)

Run by Erwyna Soo Wen Xin on Google Colab with a Tesla T4 GPU, before submission. The cells below were executed in that Colab session; this saved copy does not keep their outputs, so the outputs that run produced are saved alongside the notebook in `results/` and reproduced here unchanged.

**`results/run_8k.txt` (8K, 30 degrees):**

```
GPU: Tesla T4, compute 7.5, 40 SMs, 14.6 GB, warp 32
     max 1024 threads/block, 1024 threads/SM, 256-bit bus, peak DRAM ~320 GB/s
     copy engines: 3, concurrent kernels: yes

== Main pipelines: 7680x4320, 30.0 deg, block 16x16, 99.5 MB ==
CPU v0 nearest 509.31 ms, bilinear 1302.77 ms (one core)
variant                       H2D(ms) kernel(ms)  D2H(ms)  total(ms)  kern-SU    e2e-SU  mismatch  max
v1_nn_1d_pinned                 8.060     2.957    7.587    18.604     172.2x     27.4x          0    0
v2_nn_2d_pinned                 8.059     1.551    7.582    17.191     328.4x     29.6x          0    0
v3_bilinear_2d_pinned           8.060     2.690    7.588    18.338     484.4x     71.0x          0    0
v4_nn_2d_pageable              21.674     1.397   21.715    44.786     364.5x     11.4x          0    0
v4_bilinear_2d_pageable        21.225     2.685   20.892    44.802     485.2x     29.1x          0    0
v6_texture_bilinear_pinned      8.055     2.459    7.580    18.093     529.9x     72.0x     257964    1
kern-SU = CPU time / kernel time, e2e-SU = CPU time / (H2D + kernel + D2H)
v6 is checked against v3's CPU bilinear; small max diff is expected (8-bit weights)

== Block size sweep (kernel only, input already on GPU) ==
block    threads  occupancy  nearest(ms)  bilinear(ms)  checks
32x1          32        50%        1.517         3.383  ok
64x1          64       100%        1.446         3.368  ok
128x1        128       100%        1.444         3.379  ok
256x1        256       100%        1.445         3.370  ok
8x4           32        50%        1.729         2.954  ok
8x8           64       100%        1.589         2.935  ok
16x8         128       100%        1.520         2.829  ok
32x4         128       100%        1.462         3.854  ok
16x16        256       100%        1.453         3.337  ok
32x8         256       100%        1.565         3.875  ok
64x4         256       100%        1.510         4.004  ok
32x16        512       100%        1.631         4.321  ok
32x32       1024       100%        1.880         4.360  ok

== Streams: 4 streams, 8 chunks, bilinear, 30.0 deg ==
v5a one image serial      18.736 ms
v5a one image streams     13.777 ms  (1.36x vs serial)  mismatch 0
v5b batch of 8 serial    147.131 ms  (18.391 ms/image)
v5b batch of 8 streams    81.107 ms  (10.138 ms/image, 1.81x)  mismatch 0
```

**`results/run_streams_5deg.txt` (streams at 5 degrees):**

```
GPU: Tesla T4, compute 7.5, 40 SMs, 14.6 GB, warp 32
     max 1024 threads/block, 1024 threads/SM, 256-bit bus, peak DRAM ~320 GB/s
     copy engines: 3, concurrent kernels: yes

== Streams: 4 streams, 8 chunks, bilinear, 5.0 deg ==
v5a one image serial      19.836 ms
v5a one image streams     11.004 ms  (1.80x vs serial)  mismatch 0
v5b batch of 8 serial    145.844 ms  (18.230 ms/image)
v5b batch of 8 streams    80.484 ms  (10.060 ms/image, 1.81x)  mismatch 0
```

**Second run (sections 12 and 13), 1 Oct 2026, same Colab T4:**

```
nproc: 2
8K 7680x4320, 30.0 deg, up to 2 threads (OpenMP)
  nearest   1 thread    503.6 ms | 2 threads    470.0 ms | 1.07x, efficiency 54% | mismatch 0
  bilinear  1 thread   1246.9 ms | 2 threads   1108.3 ms | 1.13x, efficiency 56% | mismatch 0
```

```
Colab Tesla T4, 1 Oct 2026. GPU power sampled with nvidia-smi every 50 ms while
./rotate_cuda --mode main --reps 100 ran (8K, 30 degrees). 432 samples.
GPU idle 10.2 W, busy 41.1 W (374 samples above idle + 3 W)
ENERGY v2 8K: 17.16 ms/image, 0.705 J/image, 0.530 J above idle
CPU energy: RAPL not exposed on the Colab VM (not measurable)
```

Full data: `results/results_8k.csv`, `results/results_sizes.csv`, `results/results_streams_5deg.csv`, `results/results_cpu_omp.csv`, `results/results_energy.csv`.

![g1 speed-up](results/g1_speedup.png)
![g2 stage breakdown](results/g2_stage_breakdown.png)
![g3 pinned vs pageable](results/g3_pinned_vs_pageable.png)
![g4 block sweep](results/g4_block_sweep.png)
![g5 size sweep](results/g5_size_sweep.png)
![g6 streams](results/g6_streams.png)


## 1. Check the GPU and compiler

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import subprocess

# Compile for whatever GPU Colab gave us (T4 = compute capability 7.5).
try:
    cap = subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True).stdout.split()[0]
    ARCH = "sm_" + cap.replace(".", "")
except Exception:
    ARCH = "sm_75"
print("compiling for", ARCH)

## 2. Write the CUDA source

This cell is an exact copy of `rotate_cuda.cu` from our submission.

In [ ]:
%%writefile rotate_cuda.cu
/*
 * rotate_cuda.cu
 * FIT3143 Parallel Computing, Applied #2, Task 1: image rotation on a GPU
 *
 * Team:
 *   Erwyna Soo Wen Xin  36555789  esoo0013@student.monash.edu
 *   Taabish Farooq Bhat 35473932  ttaa0006@student.monash.edu
 * Task 1 (this file): Taabish Farooq Bhat
 *
 * Rotates an RGB image counterclockwise by theta about its centre using
 * R = [[cos t, -sin t], [sin t, cos t]]. Every variant below shows one
 * CUDA feature, and all GPU output is checked against the CPU output.
 *
 *   v0  CPU serial baseline (nearest and bilinear)
 *   v1  GPU naive 1D grid, nearest neighbour
 *   v2  GPU 2D grid of 2D blocks, nearest neighbour
 *   v3  GPU 2D grid, bilinear interpolation
 *   v4  v2 again but with pageable host memory (v2 uses pinned)
 *   v5  CUDA streams: (a) one image in row chunks, (b) a batch of images
 *   v6  texture object with hardware bilinear filtering
 *
 * Build (Colab T4):
 *   nvcc -O3 -arch=sm_75 --fmad=false -o rotate_cuda rotate_cuda.cu
 * --fmad=false stops nvcc fusing a*b+c into one FMA, so GPU and CPU
 * round the same way and the outputs can be compared byte for byte.
 *
 * Run:
 *   ./rotate_cuda                       8K synthetic image, 30 degrees
 *   ./rotate_cuda --w 3840 --h 2160 --angle 45 --csv results.csv
 *   ./rotate_cuda --in photo.ppm --save out     (binary P6 PPM only)
 *   ./rotate_cuda --mode blocks|streams|main|all
 */

#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <ctime>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                  \
    do {                                                                  \
        cudaError_t err_ = (call);                                        \
        if (err_ != cudaSuccess) {                                        \
            fprintf(stderr, "CUDA error \"%s\" at %s:%d\n",               \
                    cudaGetErrorString(err_), __FILE__, __LINE__);        \
            exit(EXIT_FAILURE);                                           \
        }                                                                 \
    } while (0)

// Kernel launches return nothing, so check the launch and the run apart.
#define CUDA_CHECK_LAUNCH() CUDA_CHECK(cudaGetLastError())

typedef unsigned char u8;

struct Rot {
    float c, s;   // cos(theta), sin(theta), worked out once on the host
    float cx, cy; // centre of rotation in pixel coordinates
};

/* ===================== shared host/device pixel maths ================== */

// Inverse mapping: for destination pixel (x, y) find where it came from.
// Pixel rows grow downwards, so y is flipped into Cartesian form, R^-1
// (= R^T) is applied, and y is flipped back. Worked through, that gives
// the two lines below, and the picture turns counterclockwise on screen.
__host__ __device__ inline void src_coord(const Rot r, int x, int y,
                                          float *sx, float *sy)
{
    float dx = (float)x - r.cx;
    float dy = (float)y - r.cy;
    *sx = r.cx + r.c * dx - r.s * dy;
    *sy = r.cy + r.s * dx + r.c * dy;
}

// Nearest neighbour for one output pixel: round the source point and copy.
__host__ __device__ inline void pixel_nearest(const u8 *src, u8 *dst,
                                              int w, int h, const Rot r,
                                              int x, int y)
{
    float sx, sy;
    src_coord(r, x, y, &sx, &sy);
    int ix = (int)floorf(sx + 0.5f);   // round half up, same on CPU and GPU
    int iy = (int)floorf(sy + 0.5f);
    size_t o = ((size_t)y * w + x) * 3;
    if (ix >= 0 && ix < w && iy >= 0 && iy < h) {
        size_t i = ((size_t)iy * w + ix) * 3;
        dst[o] = src[i];
        dst[o + 1] = src[i + 1];
        dst[o + 2] = src[i + 2];
    } else {
        // Corners with no source pixel become black.
        dst[o] = dst[o + 1] = dst[o + 2] = 0;
    }
}

// Pixels outside the image read as 0, which is exactly what the texture
// unit does in border mode, so v3 and v6 can be compared fairly.
__host__ __device__ inline float fetch(const u8 *src, int w, int h,
                                       int x, int y, int ch)
{
    if (x < 0 || x >= w || y < 0 || y >= h)
        return 0.0f;
    return (float)src[((size_t)y * w + x) * 3 + ch];
}

// Bilinear for one output pixel: blend the 4 nearest source pixels.
__host__ __device__ inline void pixel_bilinear(const u8 *src, u8 *dst,
                                               int w, int h, const Rot r,
                                               int x, int y)
{
    float sx, sy;
    src_coord(r, x, y, &sx, &sy);
    float fx = floorf(sx), fy = floorf(sy);
    int x0 = (int)fx, y0 = (int)fy;
    float ax = sx - fx, ay = sy - fy; // distance past the top-left neighbour
    size_t o = ((size_t)y * w + x) * 3;
    for (int ch = 0; ch < 3; ch++) {
        // Blend the 4 neighbours: 4 reads per channel instead of 1, so
        // this costs more memory traffic than nearest but has no jaggies.
        float top = fetch(src, w, h, x0, y0, ch) * (1.0f - ax)
                  + fetch(src, w, h, x0 + 1, y0, ch) * ax;
        float bot = fetch(src, w, h, x0, y0 + 1, ch) * (1.0f - ax)
                  + fetch(src, w, h, x0 + 1, y0 + 1, ch) * ax;
        float v = top * (1.0f - ay) + bot * ay;
        dst[o + ch] = (u8)(v + 0.5f);
    }
}

/* ============================== kernels ================================ */

// v1: one flat 1D grid. Simple, but each thread pays for a divide and a
// modulo, and a 256-wide block covers one long thin strip of the output.
__global__ void rotate_nn_1d(const u8 *__restrict__ src,
                             u8 *__restrict__ dst, int w, int h, Rot r)
{
    size_t id = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= (size_t)w * h)
        return; // the last block hangs past the end of the image
    int x = (int)(id % w);
    int y = (int)(id / w);
    pixel_nearest(src, dst, w, h, r, x, y);
}

// v2: 2D grid of 2D blocks. blockIdx picks the output tile, threadIdx the
// pixel inside it. No divide or modulo like v1, which is why it is
// faster. Our sweep shows shape barely matters for nearest (128x1 ~ 16x16).
// y0..y1 lets the streams version run one band of rows at a time.
__global__ void rotate_nn_2d(const u8 *__restrict__ src,
                             u8 *__restrict__ dst, int w, int h, Rot r,
                             int y0, int y1)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = y0 + blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= w || y >= y1)
        return; // grid is rounded up to whole blocks
    pixel_nearest(src, dst, w, h, r, x, y);
}

// v3: same thread layout as v2, bilinear sampling.
__global__ void rotate_bl_2d(const u8 *__restrict__ src,
                             u8 *__restrict__ dst, int w, int h, Rot r,
                             int y0, int y1)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = y0 + blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= w || y >= y1)
        return;
    pixel_bilinear(src, dst, w, h, r, x, y);
}

// v6 step 1: textures have no 3-channel 8-bit format, so pad RGB to RGBA.
__global__ void pack_rgba(const u8 *__restrict__ src, uchar4 *dst,
                          size_t pitch, int w, int h)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= w || y >= h)
        return;
    size_t i = ((size_t)y * w + x) * 3;
    uchar4 *row = (uchar4 *)((char *)dst + (size_t)y * pitch);
    row[x] = make_uchar4(src[i], src[i + 1], src[i + 2], 255);
}

// v6 step 2: the texture unit does the 4-neighbour blend in hardware and
// reads through its own cache. Its weights only have 8 fractional bits,
// so results can differ from v3 by a level or two.
__global__ void rotate_tex(cudaTextureObject_t tex, u8 *__restrict__ dst,
                           int w, int h, Rot r)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= w || y >= h)
        return;
    float sx, sy;
    src_coord(r, x, y, &sx, &sy);
    // Texel centres sit at +0.5 in unnormalised texture coordinates.
    float4 p = tex2D<float4>(tex, sx + 0.5f, sy + 0.5f);
    size_t o = ((size_t)y * w + x) * 3;
    dst[o] = (u8)(p.x * 255.0f + 0.5f);
    dst[o + 1] = (u8)(p.y * 255.0f + 0.5f);
    dst[o + 2] = (u8)(p.z * 255.0f + 0.5f);
}

/* ============================ host helpers ============================= */

static double now_ms(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1e3 + t.tv_nsec / 1e6;
}

// Milliseconds between two recorded CUDA events (GPU clock).
static float event_ms(cudaEvent_t a, cudaEvent_t b)
{
    float ms;
    CUDA_CHECK(cudaEventElapsedTime(&ms, a, b));
    return ms;
}

// A made-up but photo-sized test card: gradients, a checkerboard and thin
// grid lines, so bad interpolation or wrong direction is easy to spot.
static void make_test_image(u8 *img, int w, int h)
{
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            size_t o = ((size_t)y * w + x) * 3;
            int line = (x % 256 < 3) || (y % 256 < 3);
            int check = ((x / 64) + (y / 64)) & 1;
            img[o] = line ? 255 : (u8)(255.0 * x / (w - 1));
            img[o + 1] = line ? 255 : (u8)(255.0 * y / (h - 1));
            img[o + 2] = line ? 255 : (check ? 200 : 40);
        }
    }
    // A red block on the right edge: after +90 degrees it should be on top.
    for (int y = h / 2 - h / 16; y < h / 2 + h / 16; y++)
        for (int x = w - w / 8; x < w; x++) {
            size_t o = ((size_t)y * w + x) * 3;
            img[o] = 255;
            img[o + 1] = img[o + 2] = 0;
        }
}

// Skips whitespace and # comments in a PPM header.
static int skip_ws_comments(FILE *f)
{
    int c = fgetc(f);
    while (c == '#' || c == ' ' || c == '\n' || c == '\r' || c == '\t') {
        if (c == '#')
            while (c != '\n' && c != EOF)
                c = fgetc(f);
        c = fgetc(f);
    }
    return c;
}

// Minimal binary PPM (P6, maxval 255) reader. Returns malloc'd pixels.
static u8 *read_ppm(const char *path, int *w, int *h)
{
    FILE *f = fopen(path, "rb");
    if (!f) {
        perror(path);
        return NULL;
    }
    int maxv = 0;
    char magic[3] = {0};
    if (fread(magic, 1, 2, f) != 2 || strcmp(magic, "P6") != 0) {
        fprintf(stderr, "%s: only binary P6 PPM is supported\n", path);
        fclose(f);
        return NULL;
    }
    int vals[3];
    for (int k = 0; k < 3; k++) {
        ungetc(skip_ws_comments(f), f);
        if (fscanf(f, "%d", &vals[k]) != 1) {
            fclose(f);
            return NULL;
        }
    }
    *w = vals[0];
    *h = vals[1];
    maxv = vals[2];
    fgetc(f); // exactly one whitespace byte before the pixel data
    size_t n = (size_t)(*w) * (*h) * 3;
    u8 *img = (u8 *)malloc(n);
    if (maxv != 255 || !img || fread(img, 1, n, f) != n) {
        fprintf(stderr, "%s: bad or truncated PPM\n", path);
        free(img);
        img = NULL;
    }
    fclose(f);
    return img;
}

// Saves an RGB image as a binary P6 PPM file.
static void write_ppm(const char *path, const u8 *img, int w, int h)
{
    FILE *f = fopen(path, "wb");
    if (!f) {
        perror(path);
        return;
    }
    fprintf(f, "P6\n%d %d\n255\n", w, h);
    fwrite(img, 1, (size_t)w * h * 3, f);
    fclose(f);
}

// v0: serial CPU rotation, also the reference answer for every check.
static void rotate_cpu(const u8 *src, u8 *dst, int w, int h, Rot r,
                       int bilinear)
{
    for (int y = 0; y < h; y++)
        for (int x = 0; x < w; x++) {
            if (bilinear)
                pixel_bilinear(src, dst, w, h, r, x, y);
            else
                pixel_nearest(src, dst, w, h, r, x, y);
        }
}

struct Check {
    size_t mismatch; // bytes that differ from the CPU answer
    int max_diff;    // largest absolute difference in intensity levels
};

// Counts bytes that differ between two images and the largest difference.
static Check compare(const u8 *a, const u8 *b, size_t n)
{
    Check c = {0, 0};
    for (size_t i = 0; i < n; i++) {
        int d = abs((int)a[i] - (int)b[i]);
        if (d) {
            c.mismatch++;
            if (d > c.max_diff)
                c.max_diff = d;
        }
    }
    return c;
}

/* ====================== GPU state and one pipeline ===================== */

enum Kind { K_NN_1D, K_NN_2D, K_BL_2D, K_TEX };

struct Gpu {
    int w, h;
    size_t bytes;
    Rot r;
    dim3 block;
    u8 *d_src, *d_dst;
    uchar4 *d_rgba; // pitched RGBA copy, only used by the texture path
    size_t pitch;
    cudaTextureObject_t tex;
    int tex_ok;
    cudaEvent_t ev[4];
};

// Rounds the grid up so every pixel gets a thread.
static dim3 grid_for(dim3 block, int w, int rows)
{
    // Round up so the ragged right and bottom edges still get threads.
    return dim3((w + block.x - 1) / block.x, (rows + block.y - 1) / block.y);
}

// Launches the kernel for variant k on a band of rows in stream st.
static void launch(const Gpu &g, Kind k, dim3 block, cudaStream_t st,
                   int y0, int y1)
{
    dim3 grid = grid_for(block, g.w, y1 - y0);
    switch (k) {
    case K_NN_1D: {
        int threads = block.x * block.y;
        size_t n = (size_t)g.w * g.h;
        unsigned blocks = (unsigned)((n + threads - 1) / threads);
        rotate_nn_1d<<<blocks, threads, 0, st>>>(g.d_src, g.d_dst, g.w, g.h,
                                                 g.r);
        break;
    }
    case K_NN_2D:
        rotate_nn_2d<<<grid, block, 0, st>>>(g.d_src, g.d_dst, g.w, g.h,
                                             g.r, y0, y1);
        break;
    case K_BL_2D:
        rotate_bl_2d<<<grid, block, 0, st>>>(g.d_src, g.d_dst, g.w, g.h,
                                             g.r, y0, y1);
        break;
    case K_TEX:
        pack_rgba<<<grid, block, 0, st>>>(g.d_src, g.d_rgba, g.pitch, g.w,
                                          g.h);
        CUDA_CHECK_LAUNCH();
        rotate_tex<<<grid, block, 0, st>>>(g.tex, g.d_dst, g.w, g.h, g.r);
        break;
    }
    CUDA_CHECK_LAUNCH();
}

// v6: wraps the source in a texture object with hardware bilinear.
static void setup_texture(Gpu &g)
{
    cudaDeviceProp p;
    CUDA_CHECK(cudaGetDeviceProperties(&p, 0));
    g.tex_ok = g.w <= p.maxTexture2DLinear[0] &&
               g.h <= p.maxTexture2DLinear[1];
    if (!g.tex_ok)
        return;
    // cudaMallocPitch pads each row so it meets the texture alignment.
    CUDA_CHECK(cudaMallocPitch((void **)&g.d_rgba, &g.pitch,
                               g.w * sizeof(uchar4), g.h));
    cudaResourceDesc res;
    memset(&res, 0, sizeof(res));
    res.resType = cudaResourceTypePitch2D;
    res.res.pitch2D.devPtr = g.d_rgba;
    res.res.pitch2D.desc = cudaCreateChannelDesc<uchar4>();
    res.res.pitch2D.width = g.w;
    res.res.pitch2D.height = g.h;
    res.res.pitch2D.pitchInBytes = g.pitch;
    cudaTextureDesc td;
    memset(&td, 0, sizeof(td)); // border colour stays 0, i.e. black
    td.addressMode[0] = cudaAddressModeBorder;
    td.addressMode[1] = cudaAddressModeBorder;
    td.filterMode = cudaFilterModeLinear;       // hardware bilinear
    td.readMode = cudaReadModeNormalizedFloat;  // bytes come back as 0..1
    td.normalizedCoords = 0;                    // address in pixels
    CUDA_CHECK(cudaCreateTextureObject(&g.tex, &res, &td, NULL));
}

struct Stage {
    double h2d, ker, d2h, tot;
};

// One full pipeline: H2D copy, kernel(s), D2H copy, each stage timed with
// its own pair of CUDA events. The first run is a warm-up and is thrown
// away, then the mean of `reps` runs is returned.
static Stage run_pipeline(Gpu &g, Kind k, const u8 *h_src, u8 *h_dst,
                          int reps)
{
    Stage s = {0, 0, 0, 0};
    for (int it = 0; it <= reps; it++) {
        CUDA_CHECK(cudaEventRecord(g.ev[0]));
        CUDA_CHECK(cudaMemcpy(g.d_src, h_src, g.bytes,
                              cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(g.ev[1]));
        launch(g, k, g.block, 0, 0, g.h);
        CUDA_CHECK(cudaEventRecord(g.ev[2]));
        // Blocking copy: it waits for the kernel since both use stream 0.
        CUDA_CHECK(cudaMemcpy(h_dst, g.d_dst, g.bytes,
                              cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(g.ev[3]));
        CUDA_CHECK(cudaEventSynchronize(g.ev[3]));
        if (it == 0)
            continue;
        s.h2d += event_ms(g.ev[0], g.ev[1]);
        s.ker += event_ms(g.ev[1], g.ev[2]);
        s.d2h += event_ms(g.ev[2], g.ev[3]);
        s.tot += event_ms(g.ev[0], g.ev[3]);
    }
    s.h2d /= reps;
    s.ker /= reps;
    s.d2h /= reps;
    s.tot /= reps;
    return s;
}

// Kernel only, input already on the device. Used by the block sweep.
static double time_kernel(Gpu &g, Kind k, dim3 block, int reps)
{
    launch(g, k, block, 0, 0, g.h); // warm-up
    CUDA_CHECK(cudaEventRecord(g.ev[0]));
    for (int i = 0; i < reps; i++)
        launch(g, k, block, 0, 0, g.h);
    CUDA_CHECK(cudaEventRecord(g.ev[1]));
    CUDA_CHECK(cudaEventSynchronize(g.ev[1]));
    return event_ms(g.ev[0], g.ev[1]) / reps;
}

/* ============================ result output ============================ */

static FILE *g_csv = NULL;
static double g_angle = 30.0;

// Writes one result row to the CSV file used for the graphs.
static void csv_row(const char *section, const char *variant, int w, int h,
                    const char *block, Stage s, double cpu, double bytes,
                    Check c, double occ)
{
    if (!g_csv)
        return;
    double gb = bytes / 1e9;
    fprintf(g_csv,
            "%s,%s,%d,%d,%.1f,%s,%.4f,%.4f,%.4f,%.4f,%.3f,%.2f,%.2f,"
            "%.2f,%.2f,%.2f,%zu,%d,%.3f\n",
            section, variant, w, h, g_angle, block, s.h2d, s.ker, s.d2h,
            s.tot, cpu, s.ker > 0 ? cpu / s.ker : 0,
            s.tot > 0 ? cpu / s.tot : 0, s.h2d > 0 ? gb / (s.h2d / 1e3) : 0,
            s.d2h > 0 ? gb / (s.d2h / 1e3) : 0,
            s.ker > 0 ? 2 * gb / (s.ker / 1e3) : 0, c.mismatch, c.max_diff,
            occ);
    fflush(g_csv);
}

// Prints one row of the results table to the terminal.
static void print_row(const char *name, Stage s, double cpu, Check c)
{
    printf("%-28s %8.3f %9.3f %8.3f %9.3f %9.1fx %8.1fx %10zu %4d\n", name,
           s.h2d, s.ker, s.d2h, s.tot, cpu / s.ker, cpu / s.tot, c.mismatch,
           c.max_diff);
}

// Prints the GPU name, SM count and memory so runs can be compared.
static void print_device(void)
{
    cudaDeviceProp p;
    CUDA_CHECK(cudaGetDeviceProperties(&p, 0));
    int mem_khz = 0, bus_bits = 0;
    // Attributes instead of the deprecated cudaDeviceProp clock fields.
    CUDA_CHECK(cudaDeviceGetAttribute(&mem_khz, cudaDevAttrMemoryClockRate,
                                      0));
    CUDA_CHECK(cudaDeviceGetAttribute(&bus_bits,
                                      cudaDevAttrGlobalMemoryBusWidth, 0));
    // x2 because GDDR/HBM move data on both clock edges.
    double peak = 2.0 * mem_khz * 1e3 * (bus_bits / 8.0) / 1e9;
    printf("GPU: %s, compute %d.%d, %d SMs, %.1f GB, warp %d\n", p.name,
           p.major, p.minor, p.multiProcessorCount,
           p.totalGlobalMem / 1073741824.0, p.warpSize);
    printf("     max %d threads/block, %d threads/SM, %d-bit bus, "
           "peak DRAM ~%.0f GB/s\n",
           p.maxThreadsPerBlock, p.maxThreadsPerMultiProcessor, bus_bits,
           peak);
    printf("     copy engines: %d, concurrent kernels: %s\n\n",
           p.asyncEngineCount, p.concurrentKernels ? "yes" : "no");
}

/* =============================== sections ============================== */

static const char *HDR =
    "variant                       H2D(ms) kernel(ms)  D2H(ms)  total(ms)"
    "  kern-SU    e2e-SU  mismatch  max\n";

// v1 to v4 and v6: full pipelines against the matching CPU baseline.
static void section_main(Gpu &g, const u8 *img, const u8 *ref_nn,
                         const u8 *ref_bl, double cpu_nn, double cpu_bl,
                         int reps, const char *save)
{
    size_t n = g.bytes;
    char blk[32], blk1d[32];
    snprintf(blk, sizeof(blk), "%ux%u", g.block.x, g.block.y);
    snprintf(blk1d, sizeof(blk1d), "%u(1D)", g.block.x * g.block.y);
    u8 *pin_in, *pin_out;
    // Pinned (page-locked) buffers: the DMA engine can read them directly.
    CUDA_CHECK(cudaMallocHost((void **)&pin_in, n));
    CUDA_CHECK(cudaMallocHost((void **)&pin_out, n));
    memcpy(pin_in, img, n);
    // Pageable buffers from malloc; the driver must stage these first.
    u8 *pg_in = (u8 *)malloc(n), *pg_out = (u8 *)malloc(n);
    memcpy(pg_in, img, n);
    memset(pg_out, 0, n); // touch every page so page faults are not timed

    printf("== Main pipelines: %dx%d, %.1f deg, block %s, %.1f MB ==\n",
           g.w, g.h, g_angle, blk, n / 1e6);
    printf("CPU v0 nearest %.2f ms, bilinear %.2f ms (one core)\n", cpu_nn,
           cpu_bl);
    printf("%s", HDR);

    struct V {
        const char *name;
        Kind k;
        int pinned;
        int bl;
    } vs[] = {
        {"v1_nn_1d_pinned", K_NN_1D, 1, 0},
        {"v2_nn_2d_pinned", K_NN_2D, 1, 0},
        {"v3_bilinear_2d_pinned", K_BL_2D, 1, 1},
        {"v4_nn_2d_pageable", K_NN_2D, 0, 0},
        {"v4_bilinear_2d_pageable", K_BL_2D, 0, 1},
        {"v6_texture_bilinear_pinned", K_TEX, 1, 1},
    };
    for (size_t i = 0; i < sizeof(vs) / sizeof(vs[0]); i++) {
        if (vs[i].k == K_TEX && !g.tex_ok) {
            printf("%-28s skipped: image too big for a 2D texture\n",
                   vs[i].name);
            continue;
        }
        const u8 *in = vs[i].pinned ? pin_in : pg_in;
        u8 *out = vs[i].pinned ? pin_out : pg_out;
        memset(out, 0, n);
        CUDA_CHECK(cudaMemset(g.d_dst, 0, n)); // no stale result can pass
        Stage s = run_pipeline(g, vs[i].k, in, out, reps);
        double cpu = vs[i].bl ? cpu_bl : cpu_nn;
        Check c = compare(out, vs[i].bl ? ref_bl : ref_nn, n);
        print_row(vs[i].name, s, cpu, c);
        csv_row("main", vs[i].name, g.w, g.h,
                vs[i].k == K_NN_1D ? blk1d : blk, s, cpu,
                (double)n, c, 0);
        if (save && vs[i].k == K_BL_2D && vs[i].pinned) {
            char path[512];
            snprintf(path, sizeof(path), "%s_v3_bilinear.ppm", save);
            write_ppm(path, out, g.w, g.h);
        }
        if (save && vs[i].k == K_NN_2D && vs[i].pinned) {
            char path[512];
            snprintf(path, sizeof(path), "%s_v2_nearest.ppm", save);
            write_ppm(path, out, g.w, g.h);
        }
    }
    printf("kern-SU = CPU time / kernel time, e2e-SU = CPU time / "
           "(H2D + kernel + D2H)\n");
    printf("v6 is checked against v3's CPU bilinear; small max diff is "
           "expected (8-bit weights)\n\n");
    CUDA_CHECK(cudaFreeHost(pin_in));
    CUDA_CHECK(cudaFreeHost(pin_out));
    free(pg_in);
    free(pg_out);
}

// Block shape sweep: same work, different <<<grid, block>>> shapes.
static void section_blocks(Gpu &g, const u8 *img, const u8 *ref_nn,
                           const u8 *ref_bl, double cpu_nn, double cpu_bl,
                           int reps)
{
    const int shapes[][2] = {{32, 1}, {64, 1},  {128, 1}, {256, 1},
                             {8, 4},  {8, 8},   {16, 8},  {32, 4},
                             {16, 16}, {32, 8}, {64, 4},  {32, 16},
                             {32, 32}};
    cudaDeviceProp p;
    CUDA_CHECK(cudaGetDeviceProperties(&p, 0));
    u8 *out = (u8 *)malloc(g.bytes);
    CUDA_CHECK(cudaMemcpy(g.d_src, img, g.bytes, cudaMemcpyHostToDevice));
    printf("== Block size sweep (kernel only, input already on GPU) ==\n");
    printf("block    threads  occupancy  nearest(ms)  bilinear(ms)  "
           "checks\n");
    for (size_t i = 0; i < sizeof(shapes) / sizeof(shapes[0]); i++) {
        dim3 b(shapes[i][0], shapes[i][1]);
        int threads = b.x * b.y;
        char name[32];
        snprintf(name, sizeof(name), "%ux%u", b.x, b.y);
        double res[2];
        Check chk[2];
        double occ[2];
        for (int bl = 0; bl < 2; bl++) {
            Kind k = bl ? K_BL_2D : K_NN_2D;
            int per_sm = 0;
            // Resident blocks per SM, limited by registers, block slots
            // and threads per SM. Occupancy = resident warps / max warps.
            CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(
                &per_sm, bl ? (const void *)rotate_bl_2d
                            : (const void *)rotate_nn_2d,
                threads, 0));
            occ[bl] = (double)per_sm * threads /
                      p.maxThreadsPerMultiProcessor;
            CUDA_CHECK(cudaMemset(g.d_dst, 0, g.bytes));
            res[bl] = time_kernel(g, k, b, reps);
            CUDA_CHECK(cudaMemcpy(out, g.d_dst, g.bytes,
                                  cudaMemcpyDeviceToHost));
            chk[bl] = compare(out, bl ? ref_bl : ref_nn, g.bytes);
            Stage s = {0, res[bl], 0, 0};
            csv_row("blocks", bl ? "v3_bilinear_2d" : "v2_nn_2d", g.w, g.h,
                    name, s, bl ? cpu_bl : cpu_nn, (double)g.bytes,
                    chk[bl], occ[bl]);
        }
        printf("%-8s %7d  %8.0f%%  %11.3f  %12.3f  %s\n", name, threads,
               100 * occ[0], res[0], res[1],
               (chk[0].mismatch || chk[1].mismatch) ? "MISMATCH" : "ok");
    }
    printf("\n");
    free(out);
}

// Row range of the source that destination rows [r0, r1) can touch.
// The mapping is linear, so the extremes are at the band's corners.
static void src_rows(const Gpu &g, int r0, int r1, int *lo, int *hi)
{
    float mn = 1e30f, mx = -1e30f;
    int xs[2] = {0, g.w - 1}, ys[2] = {r0, r1 - 1};
    for (int a = 0; a < 2; a++)
        for (int b = 0; b < 2; b++) {
            float sx, sy;
            src_coord(g.r, xs[a], ys[b], &sx, &sy);
            mn = fminf(mn, sy);
            mx = fmaxf(mx, sy);
        }
    *lo = (int)floorf(mn) - 1; // -1 and +2 cover rounding and the
    *hi = (int)floorf(mx) + 2; // extra bilinear neighbour row
}

// v5a: one image, split into row chunks over several streams.
// Each output chunk may need source rows from anywhere, so a kernel only
// waits for the H2D chunks that cover its source rows. At small angles
// that is a thin band and the copies overlap well; at large angles most
// of the image is needed first and the overlap mostly comes from D2H.
// v5b: a batch of whole images, one per stream, which always overlaps.
static void section_streams(Gpu &g, const u8 *img, const u8 *ref_bl,
                            double cpu_bl, int nstreams, int nchunks,
                            int batch, int reps)
{
    size_t n = g.bytes, row = (size_t)g.w * 3;
    char blk[32];
    snprintf(blk, sizeof(blk), "%ux%u", g.block.x, g.block.y);
    u8 *pin_in, *pin_out;
    // Async copies only overlap if the host memory is pinned.
    CUDA_CHECK(cudaMallocHost((void **)&pin_in, n));
    CUDA_CHECK(cudaMallocHost((void **)&pin_out, n));
    memcpy(pin_in, img, n);

    // nstreams kernel streams plus one extra stream just for H2D copies.
    cudaStream_t *st =
        (cudaStream_t *)malloc((nstreams + 1) * sizeof(*st));
    for (int i = 0; i <= nstreams; i++)
        CUDA_CHECK(cudaStreamCreate(&st[i]));
    cudaStream_t up = st[nstreams];
    cudaEvent_t *copied = (cudaEvent_t *)malloc(nchunks * sizeof(*copied));
    for (int i = 0; i < nchunks; i++)
        CUDA_CHECK(cudaEventCreateWithFlags(&copied[i],
                                            cudaEventDisableTiming));
    int rows = (g.h + nchunks - 1) / nchunks;

    printf("== Streams: %d streams, %d chunks, bilinear, %.1f deg ==\n",
           nstreams, nchunks, g_angle);

    // Serial reference: the same bilinear pipeline with no overlap.
    Stage ser = run_pipeline(g, K_BL_2D, pin_in, pin_out, reps);

    double tot = 0;
    for (int it = 0; it <= reps; it++) {
        memset(pin_out, 0, n);
        // Events on the legacy default stream wait for all other streams.
        CUDA_CHECK(cudaEventRecord(g.ev[0], 0));
        for (int c = 0; c < nchunks; c++) {
            int r0 = c * rows, r1 = r0 + rows < g.h ? r0 + rows : g.h;
            if (r0 >= r1)
                break;
            // All H2D chunks go in order on one stream, so chunk c being
            // done means every chunk before it is done too.
            CUDA_CHECK(cudaMemcpyAsync(g.d_src + r0 * row, pin_in + r0 * row,
                                       (r1 - r0) * row,
                                       cudaMemcpyHostToDevice, up));
            CUDA_CHECK(cudaEventRecord(copied[c], up));
        }
        for (int c = 0; c < nchunks; c++) {
            int r0 = c * rows, r1 = r0 + rows < g.h ? r0 + rows : g.h;
            if (r0 >= r1)
                break;
            cudaStream_t s = st[c % nstreams];
            int lo, hi;
            src_rows(g, r0, r1, &lo, &hi);
            if (hi >= 0 && lo < g.h) {
                int last = (hi < g.h ? hi : g.h - 1) / rows;
                // Wait on the GPU, not the CPU: the host keeps queueing.
                CUDA_CHECK(cudaStreamWaitEvent(s, copied[last], 0));
            }
            launch(g, K_BL_2D, g.block, s, r0, r1);
            CUDA_CHECK(cudaMemcpyAsync(pin_out + r0 * row, g.d_dst + r0 * row,
                                       (r1 - r0) * row,
                                       cudaMemcpyDeviceToHost, s));
        }
        CUDA_CHECK(cudaEventRecord(g.ev[1], 0));
        CUDA_CHECK(cudaEventSynchronize(g.ev[1]));
        if (it > 0)
            tot += event_ms(g.ev[0], g.ev[1]);
    }
    tot /= reps;
    Check c = compare(pin_out, ref_bl, n);
    Stage s5a = {0, 0, 0, tot};
    printf("v5a one image serial   %9.3f ms\n", ser.tot);
    printf("v5a one image streams  %9.3f ms  (%.2fx vs serial)  "
           "mismatch %zu\n",
           tot, ser.tot / tot, c.mismatch);
    csv_row("streams", "v5a_serial", g.w, g.h, blk, ser, cpu_bl, (double)n,
            c, 0);
    csv_row("streams", "v5a_chunked", g.w, g.h, blk, s5a, cpu_bl,
            (double)n, c, 0);

    // v5b: batch of images. One device buffer pair per stream is enough,
    // because work inside one stream runs in order.
    u8 *b_in, *b_out;
    CUDA_CHECK(cudaMallocHost((void **)&b_in, n * batch));
    CUDA_CHECK(cudaMallocHost((void **)&b_out, n * batch));
    for (int i = 0; i < batch; i++)
        memcpy(b_in + i * n, img, n);
    u8 **d_in = (u8 **)malloc(nstreams * sizeof(u8 *));
    u8 **d_out = (u8 **)malloc(nstreams * sizeof(u8 *));
    for (int i = 0; i < nstreams; i++) {
        CUDA_CHECK(cudaMalloc((void **)&d_in[i], n));
        CUDA_CHECK(cudaMalloc((void **)&d_out[i], n));
    }
    Gpu gs = g; // copy of the launch settings, buffers swapped per stream
    double t_ser = 0, t_str = 0;
    for (int it = 0; it <= reps; it++) {
        // Serial batch: blocking copies, nothing overlaps.
        CUDA_CHECK(cudaEventRecord(g.ev[0], 0));
        for (int i = 0; i < batch; i++) {
            gs.d_src = d_in[0];
            gs.d_dst = d_out[0];
            CUDA_CHECK(cudaMemcpy(gs.d_src, b_in + i * n, n,
                                  cudaMemcpyHostToDevice));
            launch(gs, K_BL_2D, g.block, 0, 0, g.h);
            CUDA_CHECK(cudaMemcpy(b_out + i * n, gs.d_dst, n,
                                  cudaMemcpyDeviceToHost));
        }
        CUDA_CHECK(cudaEventRecord(g.ev[1], 0));
        CUDA_CHECK(cudaEventSynchronize(g.ev[1]));
        memset(b_out, 0, n * batch);
        // Streamed batch: image i goes to stream i % nstreams, so one
        // image's H2D can run beside another's kernel and a third's D2H.
        CUDA_CHECK(cudaEventRecord(g.ev[2], 0));
        for (int i = 0; i < batch; i++) {
            int k = i % nstreams;
            gs.d_src = d_in[k];
            gs.d_dst = d_out[k];
            CUDA_CHECK(cudaMemcpyAsync(gs.d_src, b_in + i * n, n,
                                       cudaMemcpyHostToDevice, st[k]));
            launch(gs, K_BL_2D, g.block, st[k], 0, g.h);
            CUDA_CHECK(cudaMemcpyAsync(b_out + i * n, gs.d_dst, n,
                                       cudaMemcpyDeviceToHost, st[k]));
        }
        CUDA_CHECK(cudaEventRecord(g.ev[3], 0));
        CUDA_CHECK(cudaEventSynchronize(g.ev[3]));
        if (it > 0) {
            t_ser += event_ms(g.ev[0], g.ev[1]);
            t_str += event_ms(g.ev[2], g.ev[3]);
        }
    }
    t_ser /= reps;
    t_str /= reps;
    Check cb = {0, 0};
    for (int i = 0; i < batch; i++) {
        Check ci = compare(b_out + i * n, ref_bl, n);
        cb.mismatch += ci.mismatch;
        if (ci.max_diff > cb.max_diff)
            cb.max_diff = ci.max_diff;
    }
    printf("v5b batch of %d serial  %9.3f ms  (%.3f ms/image)\n", batch,
           t_ser, t_ser / batch);
    printf("v5b batch of %d streams %9.3f ms  (%.3f ms/image, %.2fx)  "
           "mismatch %zu\n\n",
           batch, t_str, t_str / batch, t_ser / t_str, cb.mismatch);
    Stage sb1 = {0, 0, 0, t_ser / batch}, sb2 = {0, 0, 0, t_str / batch};
    csv_row("streams", "v5b_serial_per_image", g.w, g.h, blk, sb1, cpu_bl,
            (double)n, cb, 0);
    csv_row("streams", "v5b_streams_per_image", g.w, g.h, blk, sb2, cpu_bl,
            (double)n, cb, 0);

    for (int i = 0; i < nstreams; i++) {
        CUDA_CHECK(cudaFree(d_in[i]));
        CUDA_CHECK(cudaFree(d_out[i]));
        CUDA_CHECK(cudaStreamDestroy(st[i]));
    }
    CUDA_CHECK(cudaStreamDestroy(up));
    for (int i = 0; i < nchunks; i++)
        CUDA_CHECK(cudaEventDestroy(copied[i]));
    free(d_in);
    free(d_out);
    free(st);
    free(copied);
    CUDA_CHECK(cudaFreeHost(b_in));
    CUDA_CHECK(cudaFreeHost(b_out));
    CUDA_CHECK(cudaFreeHost(pin_in));
    CUDA_CHECK(cudaFreeHost(pin_out));
}

/* ================================= main ================================ */

int main(int argc, char **argv)
{
    int w = 7680, h = 4320, reps = 10, cpu_reps = 1;
    int bx = 16, by = 16, nstreams = 4, nchunks = 8, batch = 8;
    const char *in_path = NULL, *csv_path = NULL, *save = NULL;
    const char *mode = "all";
    for (int i = 1; i < argc; i += 2) { // options come in pairs
        const char *a = argv[i];
        const char *v = i + 1 < argc ? argv[i + 1] : NULL;
        if (!v) {
            fprintf(stderr, "missing value after %s\n", a);
            return 1;
        }
        if (!strcmp(a, "--w")) w = atoi(v);
        else if (!strcmp(a, "--h")) h = atoi(v);
        else if (!strcmp(a, "--angle")) g_angle = atof(v);
        else if (!strcmp(a, "--block")) sscanf(v, "%dx%d", &bx, &by);
        else if (!strcmp(a, "--reps")) reps = atoi(v);
        else if (!strcmp(a, "--cpu-reps")) cpu_reps = atoi(v);
        else if (!strcmp(a, "--streams")) nstreams = atoi(v);
        else if (!strcmp(a, "--chunks")) nchunks = atoi(v);
        else if (!strcmp(a, "--batch")) batch = atoi(v);
        else if (!strcmp(a, "--in")) in_path = v;
        else if (!strcmp(a, "--csv")) csv_path = v;
        else if (!strcmp(a, "--save")) save = v;
        else if (!strcmp(a, "--mode")) mode = v;
        else {
            fprintf(stderr, "unknown option %s\n", a);
            return 1;
        }
    }
    if (reps < 1) reps = 1;
    if (cpu_reps < 1) cpu_reps = 1;

    u8 *img;
    if (in_path) {
        img = read_ppm(in_path, &w, &h);
        if (!img)
            return 1;
    } else {
        img = (u8 *)malloc((size_t)w * h * 3);
        make_test_image(img, w, h);
    }
    if (save) {
        char path[512];
        snprintf(path, sizeof(path), "%s_input.ppm", save);
        write_ppm(path, img, w, h);
    }

    if (csv_path) {
        FILE *probe = fopen(csv_path, "r");
        int fresh = probe == NULL;
        if (probe)
            fclose(probe);
        g_csv = fopen(csv_path, "a"); // append so sweeps share one file
        if (g_csv && fresh)
            fprintf(g_csv, "section,variant,width,height,angle,block,"
                           "h2d_ms,kernel_ms,d2h_ms,total_ms,cpu_ms,"
                           "speedup_kernel,speedup_e2e,h2d_GBps,d2h_GBps,"
                           "kernel_GBps,mismatch,max_diff,occupancy\n");
    }

    CUDA_CHECK(cudaFree(0)); // create the CUDA context before any timing
    print_device();

    Gpu g;
    memset(&g, 0, sizeof(g));
    g.w = w;
    g.h = h;
    g.bytes = (size_t)w * h * 3;
    g.block = dim3(bx, by);
    double th = g_angle * M_PI / 180.0;
    g.r.c = (float)cos(th);
    g.r.s = (float)sin(th);
    // cos(90 deg) comes out as about 6e-17, not 0. Snap tiny values so
    // right-angle turns are exact pixel moves.
    if (fabsf(g.r.c) < 1e-7f) g.r.c = 0.0f;
    if (fabsf(g.r.s) < 1e-7f) g.r.s = 0.0f;
    g.r.cx = 0.5f * (w - 1);
    g.r.cy = 0.5f * (h - 1);

    CUDA_CHECK(cudaMalloc((void **)&g.d_src, g.bytes));
    CUDA_CHECK(cudaMalloc((void **)&g.d_dst, g.bytes));
    for (int i = 0; i < 4; i++)
        CUDA_CHECK(cudaEventCreate(&g.ev[i]));
    setup_texture(g);

    // v0: serial CPU baselines, also the reference answers for checking.
    u8 *ref_nn = (u8 *)malloc(g.bytes), *ref_bl = (u8 *)malloc(g.bytes);
    double t0 = now_ms();
    for (int i = 0; i < cpu_reps; i++)
        rotate_cpu(img, ref_nn, w, h, g.r, 0);
    double cpu_nn = (now_ms() - t0) / cpu_reps;
    t0 = now_ms();
    for (int i = 0; i < cpu_reps; i++)
        rotate_cpu(img, ref_bl, w, h, g.r, 1);
    double cpu_bl = (now_ms() - t0) / cpu_reps;
    Stage cs = {0, 0, 0, 0};
    Check none = {0, 0};
    cs.tot = cpu_nn;
    csv_row("main", "v0_cpu_nn", w, h, "-", cs, cpu_nn, (double)g.bytes,
            none, 0);
    cs.tot = cpu_bl;
    csv_row("main", "v0_cpu_bilinear", w, h, "-", cs, cpu_bl,
            (double)g.bytes, none, 0);

    int all = !strcmp(mode, "all");
    if (all || !strcmp(mode, "main"))
        section_main(g, img, ref_nn, ref_bl, cpu_nn, cpu_bl, reps, save);
    if (all || !strcmp(mode, "blocks"))
        section_blocks(g, img, ref_nn, ref_bl, cpu_nn, cpu_bl, reps);
    if (all || !strcmp(mode, "streams"))
        section_streams(g, img, ref_bl, cpu_bl, nstreams, nchunks, batch,
                        reps);

    if (g.tex_ok) {
        CUDA_CHECK(cudaDestroyTextureObject(g.tex));
        CUDA_CHECK(cudaFree(g.d_rgba));
    }
    for (int i = 0; i < 4; i++)
        CUDA_CHECK(cudaEventDestroy(g.ev[i]));
    CUDA_CHECK(cudaFree(g.d_src));
    CUDA_CHECK(cudaFree(g.d_dst));
    free(img);
    free(ref_nn);
    free(ref_bl);
    if (g_csv)
        fclose(g_csv);
    return 0;
}


## 3. Compile

`--fmad=false` stops nvcc fusing `a*b+c` into one FMA instruction, so the
GPU rounds exactly like the CPU and the outputs can be compared byte for
byte. `-Xptxas -v` prints registers per thread for each kernel, which
limits occupancy.

In [ ]:
!nvcc -O3 -arch={ARCH} --fmad=false -Xptxas -v -o rotate_cuda rotate_cuda.cu
!ls -la rotate_cuda

## 4. Main run: 8K image (7680 x 4320), 30 degrees

Runs every variant, the block sweep and the streams test. Output is saved
to `results/run_8k.txt` and `results/results_8k.csv`.

Columns: `kern-SU` = CPU time / kernel time, `e2e-SU` = CPU time /
(H2D + kernel + D2H). `mismatch` counts bytes that differ from the CPU.

In [ ]:
!mkdir -p results
!rm -f results/*.csv
!./rotate_cuda --angle 30 --csv results/results_8k.csv --save results/sample | tee results/run_8k.txt

## 5. Look at the output

Nearest neighbour copies one source pixel, so edges come out jagged.
Bilinear blends four, so they are smooth. Zoomed crops make it clear.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

imgs = [("input", "results/sample_input.ppm"),
        ("v2 nearest, +30 deg", "results/sample_v2_nearest.ppm"),
        ("v3 bilinear, +30 deg", "results/sample_v3_bilinear.ppm")]
fig, ax = plt.subplots(2, 3, figsize=(16, 7.5))
for k, (title, path) in enumerate(imgs):
    im = Image.open(path)
    ax[0, k].imshow(im.resize((im.width // 8, im.height // 8)))
    ax[0, k].set_title(title)
    ax[1, k].imshow(im.crop((3700, 1400, 3940, 1560)).resize((720, 480),
                                                             Image.NEAREST))
    ax[1, k].set_title(title + " (zoom)")
for a in ax.flat:
    a.axis("off")
plt.tight_layout()
plt.savefig("results/sample_output.png", dpi=110)
plt.show()

## 6. Image size sweep

Same pipelines from 360p to 16K. Small images cannot fill the GPU and pay
fixed launch and transfer latency, so speed-up grows with size.

In [ ]:
sizes = [(640, 360), (1280, 720), (1920, 1080), (3840, 2160),
         (7680, 4320), (15360, 8640)]
for w, h in sizes:
    print(f"--- {w} x {h} ---")
    !./rotate_cuda --w {w} --h {h} --mode main --reps 5 --csv results/results_sizes.csv | grep -E "v[0-9]|CPU"

## 7. Streams at a small angle

At 30 degrees each output band needs source rows from most of the image,
so the chunked single-image version (v5a) must wait for most of the H2D
copy. At 5 degrees each band needs a thin strip, so overlap is better.

In [ ]:
!./rotate_cuda --mode streams --angle 5 --csv results/results_streams_5deg.csv | tee results/run_streams_5deg.txt

## 8. Tables

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)
main = pd.read_csv("results/results_8k.csv")
sizes_df = pd.read_csv("results/results_sizes.csv")
st5 = pd.read_csv("results/results_streams_5deg.csv")
cols = ["variant", "block", "h2d_ms", "kernel_ms", "d2h_ms", "total_ms",
        "cpu_ms", "speedup_kernel", "speedup_e2e", "mismatch", "max_diff"]
print("8K pipelines")
display(main[main.section == "main"][cols])
print("Block sweep")
display(main[main.section == "blocks"][["variant", "block", "kernel_ms",
                                        "occupancy", "kernel_GBps",
                                        "mismatch"]])
print("Streams")
display(pd.concat([main[main.section == "streams"],
                   st5[st5.section == "streams"]])[
    ["variant", "angle", "total_ms", "mismatch"]])

## 9. Graphs (saved into `results/` for the slides)

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

H2D_C, K_C, D2H_C, CPU_C = "#D9820B", "#2E8B57", "#2F6DB5", "#7B8794"
plt.rcParams.update({"font.size": 13, "axes.spines.top": False,
                     "axes.spines.right": False})

gpu = main[(main.section == "main") & ~main.variant.str.startswith("v0")]
labels = [v.replace("_pinned", "\n(pinned)").replace("_pageable",
          "\n(pageable)") for v in gpu.variant]

# (a) speed-up over the matching CPU baseline
x = np.arange(len(gpu))
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.bar(x - 0.2, gpu.speedup_kernel, 0.4, color=K_C, label="kernel only")
ax.bar(x + 0.2, gpu.speedup_e2e, 0.4, color=D2H_C,
       label="end to end (H2D + kernel + D2H)")
for i, (a, b) in enumerate(zip(gpu.speedup_kernel, gpu.speedup_e2e)):
    ax.text(i - 0.2, a, f"{a:.0f}x", ha="center", va="bottom", fontsize=11)
    ax.text(i + 0.2, b, f"{b:.1f}x", ha="center", va="bottom", fontsize=11)
ax.set_yscale("log")
ax.set_xticks(x, labels, fontsize=10.5)
ax.set_ylabel("speed-up vs one CPU core (log)")
ax.set_title("8K rotation: kernel-only vs end-to-end speed-up")
ax.legend()
plt.tight_layout()
plt.savefig("results/g1_speedup.png", dpi=130)
plt.show()

# (b) where the GPU time goes
fig, ax = plt.subplots(figsize=(13, 5.5))
y = np.arange(len(gpu))
ax.barh(y, gpu.h2d_ms, color=H2D_C, label="H2D copy")
ax.barh(y, gpu.kernel_ms, left=gpu.h2d_ms, color=K_C, label="kernel")
ax.barh(y, gpu.d2h_ms, left=gpu.h2d_ms + gpu.kernel_ms, color=D2H_C,
        label="D2H copy")
for i, (t, k) in enumerate(zip(gpu.total_ms, gpu.kernel_ms)):
    ax.text(t, i, f"  {t:.1f} ms (kernel {100 * k / t:.0f}%)",
            va="center", fontsize=11)
ax.set_yticks(y, labels, fontsize=10.5)
ax.invert_yaxis()
ax.set_xlabel("time per 8K image (ms)")
ax.set_title("Stage breakdown per 8K image (pinned vs pageable)")
ax.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.13))
plt.tight_layout()
plt.savefig("results/g2_stage_breakdown.png", dpi=130)
plt.show()

# (c) pinned vs pageable transfer bandwidth
bw = main[main.variant.isin(["v2_nn_2d_pinned", "v4_nn_2d_pageable"])]
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(2)
ax.bar(x - 0.2, bw.h2d_GBps, 0.4, color=H2D_C, label="H2D")
ax.bar(x + 0.2, bw.d2h_GBps, 0.4, color=D2H_C, label="D2H")
for i, (a, b) in enumerate(zip(bw.h2d_GBps, bw.d2h_GBps)):
    ax.text(i - 0.2, a, f"{a:.1f}", ha="center", va="bottom")
    ax.text(i + 0.2, b, f"{b:.1f}", ha="center", va="bottom")
ax.axhline(15.75, ls="--", color=CPU_C)
ax.text(1.45, 15.9, "PCIe 3.0 x16 peak, one direction", ha="right",
        color=CPU_C, fontsize=11)
ax.set_xticks(x, ["pinned\n(cudaMallocHost)", "pageable\n(malloc)"])
ax.set_ylabel("GB/s")
ax.set_title("Host memory type vs copy bandwidth (99.5 MB image)")
ax.legend()
plt.tight_layout()
plt.savefig("results/g3_pinned_vs_pageable.png", dpi=130)
plt.show()

# (d) block size sweep
blk = main[main.section == "blocks"]
nn = blk[blk.variant == "v2_nn_2d"].reset_index(drop=True)
bl = blk[blk.variant == "v3_bilinear_2d"].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(13, 5.5))
x = np.arange(len(nn))
ax.bar(x - 0.2, nn.kernel_ms, 0.4, color=K_C, label="nearest (v2)")
ax.bar(x + 0.2, bl.kernel_ms, 0.4, color=D2H_C, label="bilinear (v3)")
ax.set_xticks(x, nn.block, rotation=30)
ax.set_xlabel("block shape (threads x by y)")
ax.set_ylabel("kernel time (ms)")
ax2 = ax.twinx()
ax2.plot(x, 100 * nn.occupancy, "o--", color=H2D_C,
         label="theoretical occupancy (v2)")
ax2.set_ylim(0, 110)
ax2.set_ylabel("occupancy (%)")
ax.set_title("Block shape sweep, 8K kernel only")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper center")
plt.tight_layout()
plt.savefig("results/g4_block_sweep.png", dpi=130)
plt.show()

# (e) image size sweep
sz = sizes_df[sizes_df.section == "main"].copy()
sz["mpix"] = sz.width * sz.height / 1e6
fig, ax = plt.subplots(figsize=(11, 5.5))
for var, col in [("v2_nn_2d_pinned", K_C), ("v3_bilinear_2d_pinned",
                                             D2H_C)]:
    d = sz[sz.variant == var]
    ax.plot(d.mpix, d.speedup_kernel, "o-", color=col,
            label=var.split("_pinned")[0] + " kernel only")
    ax.plot(d.mpix, d.speedup_e2e, "s--", color=col,
            label=var.split("_pinned")[0] + " end to end")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("image size (megapixels, log)")
ax.set_ylabel("speed-up vs one CPU core (log)")
ax.set_title("Speed-up vs image size")
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("results/g5_size_sweep.png", dpi=130)
plt.show()

# (f) streams
allst = pd.concat([main[main.section == "streams"],
                   st5[st5.section == "streams"]])
rows = []
for ang in sorted(allst.angle.unique()):
    d = allst[allst.angle == ang].set_index("variant").total_ms
    rows.append((f"one image\n{ang:.0f} deg", d["v5a_serial"],
                 d["v5a_chunked"]))
d = main[main.section == "streams"].set_index("variant").total_ms
rows.append(("batch, per image\n30 deg", d["v5b_serial_per_image"],
             d["v5b_streams_per_image"]))
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(rows))
ax.bar(x - 0.2, [r[1] for r in rows], 0.4, color=CPU_C,
       label="serial (default stream)")
ax.bar(x + 0.2, [r[2] for r in rows], 0.4, color=K_C,
       label="4 CUDA streams")
for i, r in enumerate(rows):
    ax.text(i + 0.2, r[2], f"{r[1] / r[2]:.2f}x", ha="center",
            va="bottom")
ax.set_xticks(x, [r[0] for r in rows])
ax.set_ylabel("ms per 8K image (bilinear)")
ax.set_title("Overlapping copies and compute with streams")
ax.legend()
plt.tight_layout()
plt.savefig("results/g6_streams.png", dpi=130)
plt.show()

## 10. Numbers for the slides

Pulled straight from the CSVs so the slides quote measured values.

In [ ]:
run = open("results/run_8k.txt").read()
peak = re.search(r"peak DRAM ~(\d+) GB/s", run)
peak = float(peak.group(1)) if peak else float("nan")
gpu_name = re.search(r"GPU: ([^,]+)", run).group(1)

r = main.set_index("variant")
cpu_nn = r.loc["v0_cpu_nn", "cpu_ms"]
cpu_bl = r.loc["v0_cpu_bilinear", "cpu_ms"]
v2, v3 = r.loc["v2_nn_2d_pinned"], r.loc["v3_bilinear_2d_pinned"]
v4 = r.loc["v4_nn_2d_pageable"]
copy_share = (v2.h2d_ms + v2.d2h_ms) / v2.total_ms
print(f"GPU: {gpu_name}, peak DRAM about {peak:.0f} GB/s")
print(f"CPU one core: nearest {cpu_nn:.1f} ms, bilinear {cpu_bl:.1f} ms")
print(f"v2 nearest: kernel {v2.kernel_ms:.2f} ms "
      f"({v2.speedup_kernel:.0f}x), end to end {v2.total_ms:.2f} ms "
      f"({v2.speedup_e2e:.1f}x)")
print(f"v3 bilinear: kernel {v3.kernel_ms:.2f} ms "
      f"({v3.speedup_kernel:.0f}x), end to end {v3.total_ms:.2f} ms "
      f"({v3.speedup_e2e:.1f}x)")
print(f"copies are {100 * copy_share:.0f}% of v2's GPU time")
print(f"v2 kernel moves {v2.kernel_GBps:.0f} GB/s = "
      f"{100 * v2.kernel_GBps / peak:.0f}% of peak DRAM bandwidth")
print(f"pinned H2D {v2.h2d_GBps:.1f} GB/s vs pageable {v4.h2d_GBps:.1f} "
      f"GB/s ({v2.h2d_GBps / v4.h2d_GBps:.2f}x)")
# Amdahl view: even an infinitely fast kernel cannot beat the copies.
bound = cpu_nn / (v2.h2d_ms + v2.d2h_ms)
print(f"Amdahl bound with copies kept, nearest: {bound:.1f}x "
      f"(measured {v2.speedup_e2e:.1f}x)")
bad = main[(main.section != "blocks") & (main.mismatch > 0) &
           ~main.variant.str.startswith("v6")]
print("all exact variants match the CPU" if bad.empty else bad)

## 11. Tie-back to Week 9: the same kernel in Numba

Same inverse mapping, nearest neighbour, 2D grid of 16 x 16 blocks.
Numba compiles the kernel the first time it is called (JIT), so we launch
once to warm up before timing. Wrapped in `try` so the notebook still
finishes if Numba's CUDA support is missing on the runtime.

In [ ]:
import math
import numpy as np

try:
    from numba import cuda

    @cuda.jit
    def rotate_nn(src, dst, c, s, cx, cy):
        x, y = cuda.grid(2)             # blockIdx * blockDim + threadIdx
        h, w = dst.shape[0], dst.shape[1]
        if x >= w or y >= h:
            return
        dx = np.float32(x) - cx
        dy = np.float32(y) - cy
        sx = cx + c * dx - s * dy
        sy = cy + s * dx + c * dy
        ix = int(math.floor(sx + np.float32(0.5)))
        iy = int(math.floor(sy + np.float32(0.5)))
        for ch in range(3):
            if 0 <= ix < w and 0 <= iy < h:
                dst[y, x, ch] = src[iy, ix, ch]
            else:
                dst[y, x, ch] = 0

    W, H = 3840, 2160
    t = math.radians(30)
    c, s = np.float32(math.cos(t)), np.float32(math.sin(t))
    cx, cy = np.float32((W - 1) / 2), np.float32((H - 1) / 2)
    img = np.random.randint(0, 256, (H, W, 3), dtype=np.uint8)
    d_src = cuda.to_device(img)
    d_dst = cuda.device_array_like(img)
    block = (16, 16)
    grid = ((W + 15) // 16, (H + 15) // 16)

    rotate_nn[grid, block](d_src, d_dst, c, s, cx, cy)  # JIT + warm-up
    cuda.synchronize()
    e0, e1 = cuda.event(), cuda.event()
    e0.record()
    for _ in range(10):
        rotate_nn[grid, block](d_src, d_dst, c, s, cx, cy)
    e1.record()
    e1.synchronize()
    print(f"Numba kernel, 4K: {e0.elapsed_time(e1) / 10:.3f} ms")

    # NumPy reference with the same float32 maths
    ys, xs = np.mgrid[0:H, 0:W].astype(np.float32)
    dx, dy = xs - cx, ys - cy
    ix = np.floor(cx + c * dx - s * dy + np.float32(0.5)).astype(int)
    iy = np.floor(cy + s * dx + c * dy + np.float32(0.5)).astype(int)
    ok = (ix >= 0) & (ix < W) & (iy >= 0) & (iy < H)
    ref = np.zeros_like(img)
    ref[ok] = img[iy[ok], ix[ok]]
    out = d_dst.copy_to_host()
    diff = np.count_nonzero(out != ref)
    print(f"bytes different from NumPy: {diff} of {out.size} "
          f"({100 * diff / out.size:.4f}%)")
    if diff:
        print("Numba lets LLVM fuse a*b+c into FMA, so a few pixels that"
              " sit right on a rounding edge land on the neighbour.")
except Exception as e:
    print("Numba CUDA section skipped:", e)

## 12. Fairer baseline: all CPU cores (OpenMP)

One CPU core is an unfair baseline for a GPU. This splits the same row loop
across every core with OpenMP (same maths, same rounding) and checks the
output byte for byte against the one-thread run. Colab VMs usually have
only 2 vCPUs, so `nproc` is printed with the result.

In [ ]:
%%writefile rotate_cpu_omp.c
/*
 * rotate_cpu_omp.c
 * FIT3143 Parallel Computing, Applied #2, Task 1: multi-core CPU baseline
 *
 * Team:
 *   Erwyna Soo Wen Xin  36555789  esoo0013@student.monash.edu
 *   Taabish Farooq Bhat 35473932  ttaa0006@student.monash.edu
 * Task 1 (this file): Taabish Farooq Bhat
 *
 * Same maths and rounding as v0 in rotate_cuda.cu and rotate_cpu.c, but
 * the row loop is split across all CPU cores with OpenMP. One core is an
 * unfair baseline for a GPU, so this gives the fairer comparison.
 *
 * Build: gcc -O3 -fopenmp -ffp-contract=off -o rotate_cpu_omp \
 *            rotate_cpu_omp.c -lm
 * Run:   ./rotate_cpu_omp [angle] [csv]     (default 30 degrees)
 * -ffp-contract=off keeps a*b+c as two roundings, like nvcc --fmad=false.
 */

#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#ifdef _OPENMP
#include <omp.h>
#endif

typedef unsigned char u8;

typedef struct {
    float c, s;   /* cos(theta), sin(theta) */
    float cx, cy; /* centre of rotation */
} Rot;

/* Builds cos, sin and the image centre for a rotation in degrees. */
static Rot make_rot(double deg, int w, int h)
{
    Rot r;
    double t = deg * M_PI / 180.0;
    r.c = (float)cos(t);
    r.s = (float)sin(t);
    if (fabsf(r.c) < 1e-7f) r.c = 0.0f;
    if (fabsf(r.s) < 1e-7f) r.s = 0.0f;
    r.cx = 0.5f * (w - 1);
    r.cy = 0.5f * (h - 1);
    return r;
}

/* Inverse mapping with y flipped, as in the GPU kernels. */
static void src_coord(Rot r, int x, int y, float *sx, float *sy)
{
    float dx = (float)x - r.cx;
    float dy = (float)y - r.cy;
    *sx = r.cx + r.c * dx - r.s * dy;
    *sy = r.cy + r.s * dx + r.c * dy;
}

/* Reads one channel of a source pixel, or 0 if it is off the image. */
static float fetch(const u8 *src, int w, int h, int x, int y, int ch)
{
    if (x < 0 || x >= w || y < 0 || y >= h)
        return 0.0f; /* outside counts as black */
    return (float)src[((size_t)y * w + x) * 3 + ch];
}

/* Rows are independent, so a static split of rows across threads has no
 * races: each output pixel still has exactly one writer. */
static void rotate_nearest(const u8 *src, u8 *dst, int w, int h, Rot r)
{
#pragma omp parallel for schedule(static)
    for (int y = 0; y < h; y++)
        for (int x = 0; x < w; x++) {
            float sx, sy;
            src_coord(r, x, y, &sx, &sy);
            int ix = (int)floorf(sx + 0.5f);
            int iy = (int)floorf(sy + 0.5f);
            size_t o = ((size_t)y * w + x) * 3;
            if (ix >= 0 && ix < w && iy >= 0 && iy < h) {
                memcpy(dst + o, src + ((size_t)iy * w + ix) * 3, 3);
            } else {
                dst[o] = dst[o + 1] = dst[o + 2] = 0;
            }
        }
}

/* Bilinear rotation, rows split across threads like nearest. */
static void rotate_bilinear(const u8 *src, u8 *dst, int w, int h, Rot r)
{
#pragma omp parallel for schedule(static)
    for (int y = 0; y < h; y++)
        for (int x = 0; x < w; x++) {
            float sx, sy;
            src_coord(r, x, y, &sx, &sy);
            float fx = floorf(sx), fy = floorf(sy);
            int x0 = (int)fx, y0 = (int)fy;
            float ax = sx - fx, ay = sy - fy;
            size_t o = ((size_t)y * w + x) * 3;
            for (int ch = 0; ch < 3; ch++) {
                float top = fetch(src, w, h, x0, y0, ch) * (1.0f - ax)
                          + fetch(src, w, h, x0 + 1, y0, ch) * ax;
                float bot = fetch(src, w, h, x0, y0 + 1, ch) * (1.0f - ax)
                          + fetch(src, w, h, x0 + 1, y0 + 1, ch) * ax;
                float v = top * (1.0f - ay) + bot * ay;
                dst[o + ch] = (u8)(v + 0.5f);
            }
        }
}

/* Wall-clock time in milliseconds. */
static double now_ms(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1e3 + t.tv_nsec / 1e6;
}

/* How many threads OpenMP will use (1 if built without it). */
static int max_threads(void)
{
#ifdef _OPENMP
    return omp_get_max_threads();
#else
    return 1;
#endif
}

/* Sets the OpenMP thread count for the next timed run. */
static void use_threads(int n)
{
#ifdef _OPENMP
    omp_set_num_threads(n);
#else
    (void)n;
#endif
}

/* Warm-up run, then the mean of reps runs (same method as the GPU). */
static double time_it(int bl, const u8 *src, u8 *dst, int w, int h, Rot r,
                      int reps)
{
    if (bl) rotate_bilinear(src, dst, w, h, r);
    else rotate_nearest(src, dst, w, h, r);
    double t0 = now_ms();
    for (int i = 0; i < reps; i++) {
        if (bl) rotate_bilinear(src, dst, w, h, r);
        else rotate_nearest(src, dst, w, h, r);
    }
    return (now_ms() - t0) / reps;
}

/* Times 8K rotation on 1 thread and on all threads, checks they match. */
int main(int argc, char **argv)
{
    double angle = argc > 1 ? atof(argv[1]) : 30.0;
    const char *csv = argc > 2 ? argv[2] : NULL;
    int w = 7680, h = 4320;
    size_t bytes = (size_t)w * h * 3;
    u8 *src = malloc(bytes), *ref = malloc(bytes), *dst = malloc(bytes);
    srand(3143);
    for (size_t i = 0; i < bytes; i++)
        src[i] = (u8)(rand() & 0xFF);
    Rot r = make_rot(angle, w, h);
    int nt = max_threads();

    FILE *f = csv ? fopen(csv, "w") : NULL;
    if (f)
        fprintf(f, "kernel,threads,ms,speedup_vs_1core,"
                   "parallel_efficiency,mismatch\n");
    printf("8K %dx%d, %.1f deg, up to %d threads (OpenMP)\n", w, h, angle,
           nt);
    for (int bl = 0; bl < 2; bl++) {
        const char *name = bl ? "bilinear" : "nearest";
        use_threads(1);
        double t1 = time_it(bl, src, ref, w, h, r, 3);
        use_threads(nt);
        double tn = time_it(bl, src, dst, w, h, r, 10);
        size_t bad = 0; /* bytes that differ from the one-thread result */
        for (size_t i = 0; i < bytes; i++)
            bad += ref[i] != dst[i];
        double su = t1 / tn;
        printf("  %-9s 1 thread %8.1f ms | %d threads %8.1f ms | "
               "%.2fx, efficiency %.0f%% | mismatch %zu\n",
               name, t1, nt, tn, su, 100.0 * su / nt, bad);
        if (f) {
            fprintf(f, "%s,1,%.3f,1.00,1.00,0\n", name, t1);
            fprintf(f, "%s,%d,%.3f,%.2f,%.2f,%zu\n", name, nt, tn, su,
                    su / nt, bad);
        }
    }
    if (f)
        fclose(f);
    free(src);
    free(ref);
    free(dst);
    return 0;
}


In [ ]:
!gcc -O3 -fopenmp -ffp-contract=off -o rotate_cpu_omp rotate_cpu_omp.c -lm
!nproc
!./rotate_cpu_omp 30 results/results_cpu_omp.csv | tee results/run_cpu_omp.txt

In [ ]:
import pandas as pd

omp = pd.read_csv("results/results_cpu_omp.csv")
gpu = pd.read_csv("results/results_8k.csv")
gpu = gpu[gpu.section == "main"].set_index("variant")
rows = [("nearest", "v2_nn_2d_pinned"), ("bilinear", "v3_bilinear_2d_pinned")]
for kern, var in rows:
    best = omp[omp.kernel == kern].iloc[-1]
    g = gpu.loc[var]
    print(f"{kern}: all-core CPU {best.ms:.1f} ms ({int(best.threads)} threads), "
          f"GPU kernel {g.kernel_ms:.2f} ms, GPU end to end {g.total_ms:.2f} ms")
    print(f"   GPU vs all cores: kernel {best.ms / g.kernel_ms:.0f}x, "
          f"end to end {best.ms / g.total_ms:.1f}x")

## 13. Energy per image (SCALE: account every joule)

Samples GPU board power with `nvidia-smi` every 50 ms while the main 8K
pipelines run 100 times, then multiplies the average busy power by the
measured time per image. This is SCALE step A applied to our own code.
CPU energy is read from RAPL only if the VM exposes it (Colab often does
not), so the CPU side may print as not available.

In [ ]:
import subprocess, time, glob
import pandas as pd

log = open("results/power_log.csv", "w")
mon = subprocess.Popen(
    ["nvidia-smi", "--query-gpu=power.draw,utilization.gpu",
     "--format=csv,noheader,nounits", "-lms", "50"], stdout=log)
time.sleep(3)                      # idle baseline
t_start = time.time()
subprocess.run(["./rotate_cuda", "--mode", "main", "--reps", "100",
                "--cpu-reps", "1", "--csv", "results/results_energy.csv"],
               stdout=open("results/run_energy.txt", "w"))
time.sleep(1)
mon.terminate()
log.close()

p = pd.read_csv("results/power_log.csv", header=None, names=["W", "util"])
idle_w = p.W.head(40).median()               # first ~2 s, GPU idle
busy = p[p.W > idle_w + 3]                   # samples while working
busy_w = busy.W.mean()
e = pd.read_csv("results/results_energy.csv")
e = e[e.section == "main"].set_index("variant")
t_img = e.loc["v2_nn_2d_pinned", "total_ms"] / 1000
print(f"GPU idle {idle_w:.1f} W, busy {busy_w:.1f} W ({len(busy)} samples)")
print(f"v2 nearest, 8K: {t_img*1000:.2f} ms per image -> "
      f"{busy_w * t_img:.2f} J per image "
      f"({(busy_w - idle_w) * t_img:.2f} J above idle)")

rapl = glob.glob("/sys/class/powercap/intel-rapl:0/energy_uj")
if rapl:
    def uj():
        return int(open(rapl[0]).read())
    a = uj(); t0 = time.time()
    subprocess.run(["./rotate_cpu_omp", "30"], stdout=subprocess.DEVNULL)
    b = uj(); dt = time.time() - t0
    print(f"CPU package over the OpenMP run: {(b - a) / 1e6:.1f} J in {dt:.1f} s")
else:
    print("CPU energy: RAPL not exposed on this VM (not measurable here)")
open("results/energy_summary.txt", "w").write(
    f"idle_W={idle_w:.2f} busy_W={busy_w:.2f} ms_per_image={t_img*1000:.3f} "
    f"J_per_image={busy_w*t_img:.3f}\n")

## 14. Download everything in `results/`


In [ ]:
!zip -qr results.zip results
try:
    from google.colab import files
    files.download("results.zip")
except Exception:
    print("results.zip is in the Colab file browser")